# Synthetic check — temporal routing at K = 3, 5 and 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/synthetic.ipynb)

A controlled check run through the real pipeline: a regime-switch corpus is written with the project's own frame
writer, and the models, the trainer and the metrics are the ones the real corpora use. Every clip switches between
two acoustic regimes at a random point, and the expert that transcribes it best depends on **which regimes, in which
order**, so an average over the clip's frames cannot tell the right expert, while a model of the frame sequence can.

**What this shows, and what it does not.** The construction makes the pooled summary insufficient on purpose, so the
check shows that the router *can* use temporal structure when that structure carries the routing signal: a
feasibility result. It is not evidence of an advantage under real acoustics; that evidence is the comparison with the
pooled router on the real corpora (`main_<corpus>`).

Reported per K: the WER, the selection accuracy, the gap to the oracle and the share of the best single expert's gap
to the oracle closed, for Random, the best single expert, MLP-pool, HIT-ASR and the oracle (mean ± sd over three
generator and model seeds), and the router's latency and size against K. The configuration is
`hitasr.configs.SYNTHETIC`.

| Level | What runs | Needs |
|---|---|---|
| **1** | the tables, from the record on the Hub | a CPU, seconds |
| **2** | the experiment | a GPU, ~30 min |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.configs import SYNTHETIC
from hitasr.synthetic import Synthetic

LEVEL = 1          # 1 or 2
DEVICE = "cuda"    # level 2 trains on it

synthetic = Synthetic(SYNTHETIC, level=LEVEL, device=DEVICE)

## Results

In [3]:
synthetic.results()

Synthetic regime switch: WER, selection accuracy, oracle gap, gap closed
mean ± sd over seeds (0, 1, 2); N=10000, T=128, 4 regimes. A feasibility check, not evidence under real acoustics.
  [3]
               arm    wer  wer_sd  sel_acc  sel_acc_sd  oracle_gap  gap_closed
            Random 0.2492  0.0020   0.3333      0.0000      0.2044     -0.1510
Best single expert 0.2224  0.0041   0.4228      0.0133      0.1776      0.0000
          MLP-pool 0.1710  0.0079   0.5815      0.0125      0.1261      0.2897
           HIT-ASR 0.0449  0.0027   1.0000      0.0000      0.0000      1.0000
            Oracle 0.0449  0.0027   1.0000      0.0000      0.0000      1.0000
  [5]
               arm    wer  wer_sd  sel_acc  sel_acc_sd  oracle_gap  gap_closed
            Random 0.2900  0.0012   0.2000      0.0000      0.2414     -0.1970
Best single expert 0.2506  0.0115   0.3397      0.0142      0.2020      0.0000
          MLP-pool 0.1914  0.0042   0.5133      0.0071      0.1428      0.2916
           HIT-ASR 0.0486  0.0043   1.0000      0.0000      0.0000      1.0000
            Oracle 0.0486  0.0043   1.0000      0.0000      0.0000      1.0000
  [10]
               arm    wer  wer_sd  sel_acc  sel_acc_sd  oracle_gap  gap_closed
            Random 0.3204  0.0010   0.1000      0.0000      0.2755     -0.1195
Best single expert 0.2910  0.0025   0.1845      0.0062      0.2461      0.0000
          MLP-pool 0.1976  0.0048   0.5042      0.0133      0.1528      0.3793
           HIT-ASR 0.0449  0.0033   1.0000      0.0000      0.0000      1.0000
            Oracle 0.0449  0.0033   1.0000      0.0000      0.0000      1.0000

## The router against K

In [4]:
synthetic.latency()

K,router_ms,router_ms_sd,params_M
3,2.595,0.012,3.097
5,4.090,0.027,4.019
10,7.702,0.022,6.523
